# 기능 2 — 최종 모델 선정용 비교 실험 (Aggression Detection)

> ⚠️ **이 노트북은 최종 제출 모델이 아니라, 실제로 어떤 모델을 쓸지 "정하기 위한" 비교(시뮬레이션) 과정입니다.**
> 여러 후보 모델을 같은 조건에서 돌려 보고, 그 결과를 근거로 최종 모델을 선택합니다.
> 여기서 선정된 모델의 정식 학습·평가는 `aggression_detection.ipynb` 에서 진행합니다.

기능 2(공격성 3단계 탐지)의 후보 모델을 **완전히 동일한 데이터 분할 / 동일한 Test set**으로 평가하여 공정하게 비교합니다.

- **후보 1**: TF-IDF + 로지스틱 회귀 (가벼운 머신러닝 베이스라인)
- **후보 2**: KoELECTRA-small (한국어 사전학습 딥러닝 모델)

> **공정 비교 원칙**: 두 모델 모두 `Seed=42`, `Stratified 8:1:1` 분할로 만든 **같은 Train/Test** 를 사용합니다.
> (전처리 CSV가 동일하고 분할이 결정적이므로 두 셀에서 각각 만든 분할은 완전히 같습니다.)

> **적법성**: 학습에는 Train 만 사용하고, 성능은 학습이 끝난 뒤 Test 로 1회만 측정합니다 (테스트셋 누출 없음).
> 실행 순서: 환경설정 → 데이터 검증 → 전처리 → 후보 1(TF-IDF) → 후보 2(KoELECTRA) → 최종 선정 결과
>
> 📄 이 비교 실험의 요약 결과는 별도 파일 **`test_result.txt`** 에 정리되어 있습니다.


In [ ]:
# ==========================================
# [환경 세팅] 필요한 패키지 자동 설치
# 본 노트북의 작업/실행 환경은 Google Colab(GPU 런타임) 기준입니다.
# ==========================================
import os
import sys

# 1. 필수 패키지 설치
print("[System] 필수 패키지 설치를 시작합니다...")
!pip install -q torch transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib seaborn

# 2. requirements.txt 파일 자동 생성 (제출용)
requirements_text = """
torch
transformers>=4.30.0
datasets>=2.12.0
evaluate>=0.4.0
accelerate>=0.20.0
scikit-learn>=1.2.2
pandas>=1.5.3
numpy
matplotlib
seaborn
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_text.strip())
print("[System] 제출용 requirements.txt 파일 생성 완료!")

In [ ]:
# ==========================================
# [데이터 검증] 원천 데이터셋 경로 확인
# ==========================================
DATA_PATH = "talksets-train-1_aihub.csv"  # AIHub 원본 파일명 그대로 (Colab은 /content/ 에 업로드)

if not os.path.exists(DATA_PATH):
    print("="*60)
    print(f"[오류] 현재 경로에 '{DATA_PATH}' 파일이 존재하지 않습니다.")
    print("본 실험은 AIHub 텍스트 윤리 검증 데이터셋(talksets-train-1_aihub.csv)을 사용합니다.")
    print("해당 파일을 다운로드한 후, Colab 메인 경로(/content/)에 업로드해 주세요.")
    print("="*60)
    sys.exit(1)
else:
    print(f"[System] '{DATA_PATH}' 데이터셋이 정상적으로 확인되었습니다. 전처리를 시작합니다.")

In [ ]:
import pandas as pd
import torch

if torch.cuda.is_available():
    print(f"준비 완료! GPU 사용 가능: {torch.cuda.get_device_name(0)}")
else:
    print("준비 완료! (주의) GPU가 감지되지 않았습니다 - CPU로 실행되어 학습이 매우 느릴 수 있습니다.")
    print("Colab 상단 메뉴: 런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU 로 설정하세요.")

## 1. 데이터 전처리

원천 데이터(`talksets-train-1_aihub.csv`)를 읽어 `intensity` 점수로 3단계 라벨을 만들고 정제합니다.

- **라벨링 (intensity binning)**: `< 1.0 → 0(비공격)` / `1.0 ~ 1.8 → 1(약한공격)` / `≥ 1.8 → 2(강한공격)`
- **정제**: 결측·공백·중복(text 기준) 제거
- **다운샘플링 생략**: 클래스 균등화 없이 원본 규모를 그대로 유지
- 정제 결과를 `aggression_processed_subset.csv` 로 저장하여 두 후보 모델 셀에서 동일하게 재사용합니다.

In [ ]:
def read_csv_safe(path, **kwargs):
    """인코딩(utf-8/cp949 등)을 자동으로 시도해 CSV를 읽는다.
    AIHub 원본은 CP949, 전처리 후 파일은 UTF-8이라 환경마다 다를 수 있어 모두 대응."""
    import pandas as pd
    last_err = None
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr", "latin1"):
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err

import pandas as pd

print("=== [1단계] 데이터 로드 ===")
# 원본 데이터 로드
df = read_csv_safe("talksets-train-1_aihub.csv")
print(f"전체 데이터 개수: {len(df)}개")

# intensity 점수로 3단계 label 컬럼 생성
# 임계치 기준: 1.0 미만 -> 0(비공격), 1.8 미만 -> 1(약한공격), 그 외 -> 2(강한공격)
# (NaN intensity 가 2로 잘못 분류되는 것을 막기 위해 intensity 결측을 먼저 제거)
df = df.dropna(subset=['text', 'intensity'])
df['label'] = df['intensity'].apply(lambda x: 0 if x < 1.0 else (1 if x < 1.8 else 2))

print("\n=== [2단계] Intensity Binning 완료 (3단계: 0/1/2) ===")
print("전처리 전 라벨별 데이터 분포:")
print(df['label'].value_counts().sort_index())

# 공백 및 중복 정제
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ""]
df = df.drop_duplicates(subset=['text'])

print(f"\n=== [3단계] 공백 및 중복 데이터 정제 완료 ===")
print(f"정제 후 남은 데이터 개수: {len(df)}개")

# 클래스 균등 샘플링을 생략하고 원본 규모를 그대로 유지
print(f"\n=== [4단계] 원본 규모 유지 (샘플링 생략) ===")
print(f"최종 실험용 데이터 개수: {len(df)}개")
print("최종 실험용 라벨별 데이터 분포:")
print(df['label'].value_counts().sort_index())

# 파일 저장 (두 후보 모델이 동일하게 사용)
df.to_csv("aggression_processed_subset.csv", index=False)
print("\n=== 'aggression_processed_subset.csv' 파일 저장 완료! ===")

## 2. 후보 1 — TF-IDF + 로지스틱 회귀 (베이스라인)

가벼운 머신러닝 베이스라인입니다.

- 아래 KoELECTRA 셀과 **완전히 동일한 Stratified 8:1:1 분할(Seed=42)** 을 재현하여,
  **같은 Train 으로 학습하고 같은 Test 로 평가**합니다. → 두 후보를 공정하게 비교할 수 있습니다.
- 평가 지표: Accuracy · Macro-F1 (KoELECTRA와 동일 기준)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("=== [후보 1] TF-IDF + 로지스틱 회귀 ===")

# 1. 전처리 완료된 데이터 불러오기
df = read_csv_safe("aggression_processed_subset.csv").dropna()

# 2. KoELECTRA 셀과 '동일한' Stratified 8:1:1 분할을 재현 (seed=42)
#    -> 전처리 CSV가 같고 분할이 결정적이므로 KoELECTRA 셀의 Train/Test 와 완전히 동일하다.
#    -> 베이스라인은 같은 Train 으로만 학습하고 같은 Test 로만 평가한다 (공정 비교 + 테스트셋 누출 방지).
train_df, rest_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=42, stratify=rest_df['label'])

X_train, y_train = train_df['text'], train_df['label']
X_test,  y_test  = test_df['text'],  test_df['label']
print(f"분할 -> Train: {len(X_train)}개 | Test: {len(X_test)}개 (KoELECTRA와 동일)")

# 3. TF-IDF 변환 (반드시 Train 으로만 fit)
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 4. 모델 학습
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# 5. 성능 측정 (동일 Test set)
y_pred = lr_model.predict(X_test_tfidf)
baseline_acc = accuracy_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\n[후보 1 최종 성적 / Test {len(y_test)}개]")
print(f"정확도(Accuracy): {baseline_acc:.4f}")
print(f"Macro-F1       : {baseline_f1:.4f}")
print(classification_report(y_test, y_pred, target_names=['비공격(0)', '약한공격(1)', '강한공격(2)']))

## 3. 후보 2 — KoELECTRA-small (딥러닝 메인 후보)

한국어 사전학습 모델 KoELECTRA-small 을 3-class 분류로 파인튜닝합니다.

- **분할**: 후보 1과 동일한 Stratified 8:1:1 (Seed=42)
- **학습**: Train 만 사용 / Validation 은 epoch 평가용 (테스트셋 누출 없음)
- **평가**: 학습 종료 후 Test 로 1회 측정 — Accuracy · Macro-F1 · Confusion Matrix
- 기능 3 연동을 위한 Binary(공격 vs 비공격) 지표도 함께 산출합니다.

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from datasets import Dataset

# [재현성] Seed 고정
set_seed(42)

# 1. 정제된 데이터 불러오기 (전처리 셀에서 만든 전체 데이터)
df = read_csv_safe("aggression_processed_subset.csv")
df = df.dropna()

print("=== [1] 데이터셋 분포 (정제 데이터 전체) ===")
print(df['label'].value_counts().sort_index())
print("-" * 40)

# 2. 후보 1(TF-IDF)과 '동일한' Stratified 8:1:1 분할 (seed=42)
train_df, rest_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=42, stratify=rest_df['label'])

print(f"데이터 분할 완료 -> Train: {len(train_df)}개 | Val: {len(val_df)}개 | Test: {len(test_df)}개")

# HuggingFace Dataset 변환
train_dataset = Dataset.from_pandas(train_df.drop(columns=['__index_level_0__'], errors='ignore'))
val_dataset = Dataset.from_pandas(val_df.drop(columns=['__index_level_0__'], errors='ignore'))
test_dataset = Dataset.from_pandas(test_df.drop(columns=['__index_level_0__'], errors='ignore'))

# 토크나이저 및 모델 세팅
MODEL_NAME = "monologg/koelectra-small-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,   # 체크포인트 1개만 유지 (디스크 정리)
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # 학습은 Train 만 사용
    eval_dataset=tokenized_val,      # Validation 은 epoch 평가용
    compute_metrics=compute_metrics,
)

print("\nKoELECTRA-small 학습 시작 (Seed=42)...")
trainer.train()

print("\n" + "="*50)
print("=== [최종 평가] KoELECTRA - TEST 데이터셋(10%) 성능 ===")
print("="*50)

test_predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(test_predictions.predictions, axis=-1)
y_true = test_predictions.label_ids

# 후보 비교용 핵심 지표 저장
koelectra_acc = accuracy_score(y_true, y_pred)
koelectra_f1 = f1_score(y_true, y_pred, average="macro")

print("[3-Class 정밀 분류 리포트 (Test Dataset)]")
print(f"정확도(Accuracy): {koelectra_acc:.4f}")
print(f"Macro-F1       : {koelectra_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=['비공격(0)', '약한공격(1)', '강한공격(2)']))
print("-" * 40)

# Binary 지표 산출 (기능 3 연동용)
y_true_binary = np.where(y_true == 0, 0, 1)
y_pred_binary = np.where(y_pred == 0, 0, 1)

print("\n=== 기능 3 연동용 Binary 성능 지표 (공격 vs 비공격) ===")
print(f"Binary Accuracy : {accuracy_score(y_true_binary, y_pred_binary):.4f}")
print(f"Binary Precision: {precision_score(y_true_binary, y_pred_binary):.4f}")
print(f"Binary Recall   : {recall_score(y_true_binary, y_pred_binary):.4f}")
print(f"Binary F1-Score : {f1_score(y_true_binary, y_pred_binary):.4f}")
print("-" * 40)

# Confusion Matrix 시각화
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Non-Agg (0)', 'Weak-Agg (1)', 'Strong-Agg (2)'],
            yticklabels=['Non-Agg (0)', 'Weak-Agg (1)', 'Strong-Agg (2)'])
plt.title('Confusion Matrix - KoELECTRA (Test Dataset)', fontsize=14, pad=15)
plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
plt.ylabel('True Label', fontsize=12, labelpad=10)
plt.show()

# --------------------------------------------------
# 실제 inference 예시 (총 10개 문장)
# --------------------------------------------------
additional_comments = [
    "아 진짜 짜증나네 일 똑바로 안 하냐? 죽고 싶냐 진짜",
    "와 미친... 이거 진짜 대박이다 소름 돋았음 ㅠㅠ",
    "적당히 좀 하세요 진짜 짜증나게 만드네 ㅡㅡ",
    "야 이 개새끼야 대가리 총 맞았냐? 걍 뒤져라",
    "그쪽 의견도 알겠는데 제 생각은 좀 달라요.",
    "저 지금 알바 중이라 확인이 늦었어요 ㅠㅠ 죄송해요",
    "와 피피티 퀄리티 진짜 미쳤다!! 너무 고생하셨어요!! 😭👍",
    "그쪽은 저번 주부터 피드백도 없고... 참여 안 하시는 걸로 알게요 ^^",
    "제출 기한이 오늘 자정까지인데, 혹시 자료조사는 언제쯤 주실 수 있을까요?",
    "자료조사 다 했다고 보내신 게 고작 이건가요? 초등학생도 안 하겠네"
]

print("\n=== 실제 inference 예시 결과 ===")
model.eval()
for i, test_comment in enumerate(additional_comments, 1):
    inputs = tokenizer(test_comment, return_tensors="pt", padding=True, truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_label = torch.argmax(outputs.logits, dim=-1).item()
    label_dict = {0: "비공격 (0)", 1: "약한 공격 (1)", 2: "강한 공격 (2)"}
    print(f"예시 {i}) 입력 댓글: \"{test_comment}\" -> AI 예측 결과: {label_dict[pred_label]}")

## 4. 최종 모델 선정 결과

두 후보를 **동일한 Test set / 동일한 지표(Accuracy · Macro-F1)** 로 비교하여 최종 모델을 선정합니다.
(아래 셀은 후보 1·후보 2 셀을 모두 실행한 뒤 돌려야 합니다.)

In [ ]:
# ==========================================
# 최종 모델 선정: 두 후보 비교
# (후보 1 / 후보 2 셀을 모두 실행한 뒤 돌릴 것)
# ==========================================
print("=" * 55)
print(" 기능 2 - 최종 모델 선정 결과 (동일 Test set / 동일 지표)")
print("=" * 55)
print(f"{'후보':<26}{'Accuracy':>10}{'Macro-F1':>10}")
print("-" * 55)
print(f"{'후보1) TF-IDF + 로지스틱회귀':<22}{baseline_acc:>10.4f}{baseline_f1:>10.4f}")
print(f"{'후보2) KoELECTRA-small':<24}{koelectra_acc:>10.4f}{koelectra_f1:>10.4f}")
print("-" * 55)

# 클래스 불균형을 고려해 Macro-F1 을 1차 기준으로 최종 모델 선정
if koelectra_f1 >= baseline_f1:
    winner = "KoELECTRA-small"
    w_acc, w_f1 = koelectra_acc, koelectra_f1
else:
    winner = "TF-IDF + 로지스틱 회귀"
    w_acc, w_f1 = baseline_acc, baseline_f1

print(f"\n✅ 최종 선정 모델: {winner}")
print(f"   (Macro-F1 {w_f1:.4f} / Accuracy {w_acc:.4f} 기준)")
print("   * 클래스 불균형을 고려해 Macro-F1 을 1차 선정 기준으로 사용했습니다.")